 ML CHALLENGE

The first thing that we noticed in the Training Dataset was that the class labels were already provided so we were working on a supervised learning problem. This generally means that algorithms like K-Means clustering would likely work worse than algorithms like logistic regression. 

We explored the dataset using `describe()` on the training data. We observed that the maximum values were consistently much larger than the mean, median, and the 75th percentile.

When examining higher percentiles (90th, 95th, and 98th), the presence of outliers was clear after 95th percentile.To make sure we plotted boxplots for several features and the outliers were again significant in almost all the features.

So we could proceed in two ways , process the data for the outliers and then test algorithms on it or use the algorithms that are least affected by outliers. Random Forest is famously robust to outliers so we decided to test it first to determine if we are going in the right direction. 

To actually test our models we used train_test_split where we split 20 percent of our training data to be the test data and 80 percent to be the training data. To make sure we test the models exhaustively we ran it through different splits of train_test.

Surprisingly just the default `RandomForestClassifier()` gave a F1-score of about 0.98 which is already a very good F1 score. Then we tried removing the columns with low feature importance which made negligible changes. At last after some hyperparameter tuning n_estimators=300 gave the best result (F1 score around 98.2)

We decided to check for Logistic Regression too with the thinking that maybe we could make a voting ensemble combining both random forest and Logistic Regression but the basic Logistic Regression model gave an F1-score of around 0.70 which was already too low compared to our previous one to try to improve.

We decided to improve on the original random forest only . We therefore explored boosting algorithms, which often outperform Random Forest on tabular datasets because they build trees sequentially using gradient boosting. Our options were XGBoost , CatBoost , LightGBM. XGBoost after some hyperparameter tuning gave a F1-score of around 98.5 . CatBoost stayed below 97 so we decided to not go with it.Then we tested LightGBM which with not much tuning was already giving a F1-score of around 98.4. We spent some time tuning it and it led to F1-score reaching 98.6-98.7 consistently. We tried tuning both XGBoost and LightGBM more but it didnt improve after this.

Hence we used the LightGBM model as our final model.



In [44]:
# Install lightGBM if not already installed (--break-system-packages needed for arch linux to use pip)
!pip install lightgbm --break-system-packages

Defaulting to user installation because normal site-packages is not writeable


In [45]:
# pandas is used for data manipulation and reading csv files
import pandas as pd
from lightgbm import LGBMClassifier

In [37]:
# Load the training dataset
# X contains all feature columns except class
# y contains the class labels used for training 
train = pd.read_csv("TRAIN.csv")
X=train.drop("Class",axis=1)
y=train["Class"]

In [46]:
# Load the testing dataset
# test_ids holds the id of all the rows which will be required in the final.csv file to map the predictions to correct rows
# X_test contains the feature data on which the trained model will make predictions (the ID column is removed)
test = pd.read_csv("TEST.csv")
test_ids = test["ID"]
X_test = test.drop(columns=["ID"])

In [47]:
# This is the LightGBM model made using the LGBMClassifier
# The parameters were selected through experimentation and trial-and-error
# subsample and colsample_bytree handles how much data every tree is allowed to see reducing overfitting.
# verbose=-1 was added just to keep the output clean
# # num_leaves was increased since the dataset is relatively small, allowing the model to learn more complex patterns

model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.3,
    max_depth=9,
    subsample=0.8,
    colsample_bytree=0.8,
    num_leaves=120,
    verbose=-1
)

model.fit(X, y)

,boosting_type,'gbdt'
,num_leaves,120
,max_depth,9
,learning_rate,0.3
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [48]:
# This makes a final table and adds the original ids with the predictions labelled as class as the criterion for final file
predictions = model.predict(X_test)
final = pd.DataFrame({
    "ID": test_ids,
    "CLASS": predictions
})

In [33]:
final.to_csv("FINAL.csv", index=False)